<a href="https://colab.research.google.com/github/Jouwana-Daibes/Comprehension-and-Grammar-evaluation-experiments/blob/main/USE_SBERT_COMPARISON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install supabase
!pip install supabase sentence-transformers tensorflow-hub tensorflow
!pip install psycopg2-binary sqlalchemy
!pip install sentencepiece tf_sentencepiece
!pip install tensorflow-text
!pip3 install tensorflow_text>=2.0.0rc0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.4 MB/s eta 0:00:00


In [ ]:
import tensorflow_hub as hub
import tensorflow.compat.v1 as tf
import numpy as np
import tensorflow_text # Import tensorflow_text to register SentencepieceOp
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer

# Supabase connection
url = "https://jtsxhsyospghskalmbfx.supabase.co"
key = "sb_publishable_U3Sz7aE-OfPS6Q_-EssgUg_cUGN3qBV"
supabase: Client = create_client(url, key)

# -------------------------
# Load models (once)
# -------------------------

use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder-multilingual/3")
def use_embed(texts):
    return use_model(texts).numpy()

# List of Sentence-BERT style models to try (multilingual + others)
sbert_model_ids = [
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "sentence-transformers/LaBSE",
    "sentence-transformers/distiluse-base-multilingual-cased-v1",
    "akhooli/Arabic-SBERT-100K",
    "Omartificial-Intelligence-Space/mmbert-base-arabic-nli",
    "NAMAA-Space/AraModernBert-Base-STS",
    "Omartificial-Intelligence-Space/Arabic-MiniLM-L12-v2-all-nli-triplet",
    "Omartificial-Intelligence-Space/Arabic-mpnet-base-all-nli-triplet",
    "medmediani/Arabic-KW-Mdel"
]

# Load SBERT models into a dict
sbert_models = {}
for mid in sbert_model_ids:
    try:
        sbert_models[mid] = SentenceTransformer(mid)
    except Exception as e:
        print(f"Warning: failed to load {mid}: {e}")

def sbert_embed(model_id, texts):
    model = sbert_models[model_id]
    return model.encode(texts)

# -------------------------
# Utility
# -------------------------
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# -------------------------
# Multi-model check function
# -------------------------
def check_response_all_models(question_id, my_response, thresholds=None, verbose=True):
    """
    For the given question_id and response, compute similarities using:
      - USE
      - each SBERT model in sbert_models
    Print detailed comparisons per model and return a dict of results.
    thresholds: optional dict mapping model name -> threshold (default 0.65)
    """
    if thresholds is None:
        thresholds = {}

    # Fetch reference answers
    answers = supabase.table("answers").select("text").eq("question_id", question_id).execute().data
    answer_texts = [a["text"] for a in answers]

    results = {}

    # --- USE
    try:
        resp_emb = use_embed([my_response])[0]
        ref_embs = use_embed(answer_texts)
        scores = [cosine_similarity(resp_emb, ref) for ref in ref_embs]
        max_score = max(scores) if scores else 0
        thr = thresholds.get("USE", 0.65)
        decision = max_score > thr

        if verbose:
          print(f"\n=== Model: USE (universal-sentence-encoder-multilingual/3) ===")
          print(f"Comparisons for response: {my_response}")
          for i, ref in enumerate(answer_texts):
            print(f"Reference answer {i+1}: {ref}")
            print(f"  USE similarity:   {scores[i]:.3f}")
          print(f"Final decision (USE): {'Correct' if decision else 'Incorrect'} (score={max_score:.3f})")

        results["USE"] = {"scores": scores, "max": max_score, "decision": decision}
    except Exception as e:
        print("USE error:", e)
        results["USE"] = {"error": str(e)}

    # --- SBERT models
    for mid, model in sbert_models.items():
        try:
            model = sbert_models[mid]
            resp_emb = model.encode(my_response)
            ref_embs = model.encode(answer_texts)
            scores = [cosine_similarity(resp_emb, ref) for ref in ref_embs]
            max_score = max(scores) if scores else 0
            thr = thresholds.get(mid, 0.65)
            decision = max_score > thr

            if verbose:
              print(f"\n=== Model: {mid} ===")
              print(f"Comparisons for response: {my_response}")
              for i, ref in enumerate(answer_texts):
                print(f"Reference answer {i+1}: {ref}")
                print(f"  SBERT similarity: {scores[i]:.3f}")
              print(f"Final decision ({mid}): {'Correct' if decision else 'Incorrect'} (score={max_score:.3f})")

            results[mid] = {"scores": scores, "max": max_score, "decision": decision}
        except Exception as e:
            print(f"Error with model {mid}: {e}")
            results[mid] = {"error": str(e)}

    return results
"""
import csv

def evaluate_all_models(
    thresholds=None,
    csv_filename="evaluation_results.csv"
):

    if thresholds is None:
        thresholds = {}

    # =====================================================
    # LOAD ALL RESPONSES TABLE (PAGINATION SAFE)
    # =====================================================
    responses = []

    batch_size = 1000
    start = 0

    while True:

        batch = (
            supabase
            .table("responses")
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
            .data
        )

        if not batch:
            break

        responses.extend(batch)

        print(f"Loaded {len(responses)} responses...")

        start += batch_size

    # =====================================================
    # LOAD ALL ANSWERS TABLE (PAGINATION SAFE)
    # =====================================================
    answers_all = []

    start = 0

    while True:

        batch = (
            supabase
            .table("answers")
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
            .data
        )

        if not batch:
            break

        answers_all.extend(batch)

        print(f"Loaded {len(answers_all)} answers...")

        start += batch_size

    # =====================================================
    # GROUP ANSWERS BY QUESTION ID
    # =====================================================
    answers_by_qid = {}

    for a in answers_all:

        qid = a["question_id"]

        if qid not in answers_by_qid:
            answers_by_qid[qid] = []

        answers_by_qid[qid].append(a["text"])

    # =====================================================
    # MODEL NAMES
    # =====================================================
    model_names = ["USE"] + list(sbert_models.keys())

    # =====================================================
    # CREATE CSV FILE
    # =====================================================
    headers = [

        "question_id",
        "response",
        "reference_answer"
    ] + model_names

    csv_file = open(
        csv_filename,
        mode="w",
        newline="",
        encoding="utf-8"
    )

    writer = csv.writer(csv_file)

    # Write header row
    writer.writerow(headers)

    # =====================================================
    # METRICS STORAGE
    # =====================================================
    metrics = {}

    for m in model_names:
        metrics[m] = {
            "TP": 0,
            "FP": 0,
            "TN": 0,
            "FN": 0
        }

    # =====================================================
    # MAIN EVALUATION LOOP
    # =====================================================
    for i, row in enumerate(responses):

        if i % 50 == 0:
            print(f"Processed {i}/{len(responses)}")

        qid = row["question_id"]
        response_text = row["response_text"]
        true_label = row["meaning_label"]

        # Get all reference answers
        reference_answers = answers_by_qid.get(qid, [])

        if not reference_answers:
            continue

        # =================================================
        # USE MODEL
        # =================================================
        use_scores = []

        try:

            resp_emb_use = use_embed([response_text])[0]

            ref_embs_use = use_embed(reference_answers)

            use_scores = [
                cosine_similarity(resp_emb_use, ref_emb)
                for ref_emb in ref_embs_use
            ]

            # Metrics
            max_score = max(use_scores) if use_scores else 0

            threshold = thresholds.get("USE", 0.65)

            pred = max_score > threshold

            if pred and true_label:
                metrics["USE"]["TP"] += 1

            elif pred and not true_label:
                metrics["USE"]["FP"] += 1

            elif not pred and not true_label:
                metrics["USE"]["TN"] += 1

            else:
                metrics["USE"]["FN"] += 1

        except Exception as e:

            print(f"USE error for question {qid}: {e}")

            use_scores = [None] * len(reference_answers)

        # =================================================
        # SBERT MODELS
        # =================================================
        sbert_scores = {}

        for model_name, model in sbert_models.items():

            try:

                resp_emb = model.encode(response_text)

                ref_embs = model.encode(reference_answers)

                scores = [
                    cosine_similarity(resp_emb, ref_emb)
                    for ref_emb in ref_embs
                ]

                sbert_scores[model_name] = scores

                # Metrics
                max_score = max(scores) if scores else 0

                threshold = thresholds.get(model_name, 0.65)

                pred = max_score > threshold

                if pred and true_label:
                    metrics[model_name]["TP"] += 1

                elif pred and not true_label:
                    metrics[model_name]["FP"] += 1

                elif not pred and not true_label:
                    metrics[model_name]["TN"] += 1

                else:
                    metrics[model_name]["FN"] += 1

            except Exception as e:

                print(f"Error with model {model_name}: {e}")

                sbert_scores[model_name] = [None] * len(reference_answers)

        # =================================================
        # WRITE CSV ROWS
        # One row per reference answer
        # =================================================
        for idx, ref_answer in enumerate(reference_answers):

            row_data = [
                qid,
                response_text,
                ref_answer,
                true_label
            ]

            # USE score
            row_data.append(
                use_scores[idx]
                if idx < len(use_scores)
                else None
            )

            # SBERT scores
            for model_name in sbert_models.keys():

                scores = sbert_scores.get(model_name, [])

                row_data.append(
                    scores[idx]
                    if idx < len(scores)
                    else None
                )

            writer.writerow(row_data)

    # =====================================================
    # CLOSE CSV
    # =====================================================
    csv_file.close()

    print(f"\nSaved similarity results to: {csv_filename}")

    # =====================================================
    # FINAL METRICS
    # =====================================================
    print("\n" + "=" * 60)
    print("FINAL METRICS")
    print("=" * 60)

    for model_name, m in metrics.items():

        TP = m["TP"]
        FP = m["FP"]
        TN = m["TN"]
        FN = m["FN"]

        total = TP + FP + TN + FN

        accuracy = (TP + TN) / total if total else 0

        precision = TP / (TP + FP) if (TP + FP) else 0

        recall = TP / (TP + FN) if (TP + FN) else 0

        f1 = (
            (2 * precision * recall) / (precision + recall)
            if (precision + recall)
            else 0
        )

        print(f"\n=== {model_name} ===")
        print(f"Accuracy : {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")
        print(f"F1 Score : {f1:.4f}")
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'\nimport csv\n\ndef evaluate_all_models(\n    thresholds=None,\n    csv_filename="evaluation_results.csv"\n):\n\n    if thresholds is None:\n        thresholds = {}\n\n    # =====================================================\n    # LOAD ALL RESPONSES TABLE (PAGINATION SAFE)\n    # =====================================================\n    responses = []\n\n    batch_size = 1000\n    start = 0\n\n    while True:\n\n        batch = (\n            supabase\n            .table("responses")\n            .select("*")\n            .range(start, start + batch_size - 1)\n            .execute()\n            .data\n        )\n\n        if not batch:\n            break\n\n        responses.extend(batch)\n\n        print(f"Loaded {len(responses)} responses...")\n\n        start += batch_size\n\n    # =====================================================\n    # LOAD ALL ANSWERS TABLE (PAGINATION SAFE)\n    # =====================================================\n    answers_all = []\n\n    start

In [ ]:
import csv
import numpy as np

def load_all_data():

    # -------------------------
    # RESPONSES
    # -------------------------
    responses = []

    batch_size = 1000
    start = 0

    while True:

        batch = (
            supabase
            .table("responses")
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
            .data
        )

        if not batch:
            break

        responses.extend(batch)

        print(f"Loaded {len(responses)} responses")

        start += batch_size

    # -------------------------
    # ANSWERS
    # -------------------------
    answers_all = []

    start = 0

    while True:

        batch = (
            supabase
            .table("answers")
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
            .data
        )

        if not batch:
            break

        answers_all.extend(batch)

        print(f"Loaded {len(answers_all)} answers")

        start += batch_size

    # -------------------------
    # GROUP ANSWERS
    # -------------------------
    answers_by_qid = {}

    for a in answers_all:

        qid = a["question_id"]

        if qid not in answers_by_qid:
            answers_by_qid[qid] = []

        answers_by_qid[qid].append(a["text"])

    # -------------------------
    # QUESTIONS
    # -------------------------
    questions = []

    start = 0

    while True:

        batch = (
            supabase
            .table("questions")
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
            .data
        )

        if not batch:
            break

        questions.extend(batch)


        print(f"Loaded {len(questions)} questions")

        start += batch_size
    questions_by_qid = {}

    for q in questions:

        questions_by_qid[q["id"]] = q

    return responses, answers_by_qid, questions_by_qid


In [ ]:
def precompute_use_embeddings(answers_by_qid):

    use_ref_embeddings = {}

    for qid, refs in answers_by_qid.items():

        use_ref_embeddings[qid] = use_embed(refs)

    return use_ref_embeddings

def precompute_sbert_embeddings(model_name, answers_by_qid):

    model = sbert_models[model_name]

    ref_embeddings = {}

    for qid, refs in answers_by_qid.items():

        ref_embeddings[qid] = model.encode(refs)

    return ref_embeddings

In [ ]:
def evaluate_use(
    responses,
    answers_by_qid,
    ref_embeddings,
    questions_by_qid,
    csv_filename="use_results.csv",
    threshold=0.65,
):

    headers = [
        "response_id",
        "story_id",
        "question_id",
        "question_text",
        "response_text",
        "best_reference_answer",
        "similarity",
        "true_label"
    ]

    with open(
        csv_filename,
        mode="w",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.writer(f)
        writer.writerow(headers)

        for i, row in enumerate(responses):

            if i % 50 == 0:
                print(f"USE: {i}/{len(responses)}")

            qid = row["question_id"]
            question_info = questions_by_qid.get(qid, {})

            question_text = question_info.get("text")
            story_id = question_info.get("story_id")

            refs = answers_by_qid.get(qid, [])

            if not refs:
                continue

            response_text = row["response_text"]

            resp_emb = use_embed([response_text])[0]

            ref_embs = ref_embeddings[qid]

            scores = [
                cosine_similarity(resp_emb, ref_emb)
                for ref_emb in ref_embs
            ]

            best_idx = int(np.argmax(scores))

            writer.writerow([
                row.get("id"),
                story_id,
                qid,
                question_text,
                response_text,
                refs[best_idx],
                scores[best_idx],
                row.get("meaning_label")
            ])

    print(f"Saved USE results to {csv_filename}")

In [ ]:
def evaluate_sbert_model(
    model_name,
    responses,
    answers_by_qid,
    ref_embeddings,
    csv_filename,
    questions_by_qid,
    threshold=0.65
):

    model = sbert_models[model_name]

    headers = [
        "response_id",
        "story_id",
        "question_id",
        "question_text",
        "response_text",
        "best_reference_answer",
        "similarity",
        "true_label"
    ]

    with open(
        csv_filename,
        mode="w",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        writer = csv.writer(f)

        writer.writerow(headers)

        for i, row in enumerate(responses):

            if i % 50 == 0:
                print(f"{model_name}: {i}/{len(responses)}")

            qid = row["question_id"]
            question_info = questions_by_qid.get(qid, {})

            question_text = question_info.get("text")
            story_id = question_info.get("story_id")

            refs = answers_by_qid.get(qid, [])

            if not refs:
                continue

            response_text = row["response_text"]

            resp_emb = model.encode(response_text)

            ref_embs = ref_embeddings[qid]

            scores = [
                cosine_similarity(resp_emb, ref_emb)
                for ref_emb in ref_embs
            ]

            best_idx = int(np.argmax(scores))

            writer.writerow([
                row.get("id"),
                story_id,
                qid,
                question_text,
                response_text,
                refs[best_idx],
                scores[best_idx],
                row.get("meaning_label")
            ])

    print(f"Saved {model_name} results to {csv_filename}")

In [ ]:
# ==========================================
# LOAD DATA ONCE
# ==========================================

responses, answers_by_qid, questions_by_qid = load_all_data()

# ==========================================
# USE
# ==========================================

use_ref_embeddings = precompute_use_embeddings(
    answers_by_qid
)

evaluate_use(
    responses,
    answers_by_qid,
    use_ref_embeddings,
    questions_by_qid,
    csv_filename="use_results.csv"
)

# ==========================================
# SBERT MODELS
# ==========================================

for model_name in sbert_models.keys():

    print(f"\nPreparing embeddings for {model_name}")

    ref_embeddings = precompute_sbert_embeddings(
        model_name,
        answers_by_qid
    )

    safe_name = model_name.replace("/", "_")

    evaluate_sbert_model(
        model_name=model_name,
        responses=responses,
        answers_by_qid=answers_by_qid,
        ref_embeddings=ref_embeddings,
        questions_by_qid=questions_by_qid,
        csv_filename=f"{safe_name}.csv"
    )

Loaded 1000 responses
Loaded 1450 responses
Loaded 155 answers
Loaded 52 questions
USE: 0/1450
USE: 50/1450
USE: 100/1450
USE: 150/1450
USE: 200/1450
USE: 250/1450
USE: 300/1450
USE: 350/1450
USE: 400/1450
USE: 450/1450
USE: 500/1450
USE: 550/1450
USE: 600/1450
USE: 650/1450
USE: 700/1450
USE: 750/1450
USE: 800/1450
USE: 850/1450
USE: 900/1450
USE: 950/1450
USE: 1000/1450
USE: 1050/1450
USE: 1100/1450
USE: 1150/1450
USE: 1200/1450
USE: 1250/1450
USE: 1300/1450
USE: 1350/1450
USE: 1400/1450
Saved USE results to use_results.csv

Preparing embeddings for sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: 0/1450
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: 50/1450
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: 100/1450
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: 150/1450
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: 200/1450
sentence-transformers/paraphrase-mu

In [ ]:
print(responses[0].keys())

dict_keys(['id', 'question_id', 'response_text', 'meaning_label', 'language_label'])


In [ ]:
"""
def evaluate_all_models(thresholds=None):
    if thresholds is None:
        thresholds = {}

    # -------------------------
    # Load data ONCE
    # -------------------------
    responses = supabase.table("responses").select("*").execute().data
    answers_all = supabase.table("answers").select("*").execute().data

    # Group answers by question_id
    answers_by_qid = {}
    for a in answers_all:
        answers_by_qid.setdefault(a["question_id"], []).append(a["text"])

    # -------------------------
    # Metrics storage
    # -------------------------
    metrics = {}

    # Initialize models list
    model_names = ["USE"] + list(sbert_models.keys())
    for m in model_names:
        metrics[m] = {"TP": 0, "FP": 0, "TN": 0, "FN": 0}

    # -------------------------
    # Evaluation loop
    # -------------------------
    for i, row in enumerate(responses):

        if i % 50 == 0:
            print(f"Processed {i}/{len(responses)}")

        qid = row["question_id"]
        text = row["response_text"]
        true_label = row["meaning_label"]

        answer_texts = answers_by_qid.get(qid, [])
        if not answer_texts:
            continue

        # -------------------------
        # USE model
        # -------------------------
        try:
            resp_emb = use_embed([text])[0]
            ref_embs = use_embed(answer_texts)

            scores = [cosine_similarity(resp_emb, ref) for ref in ref_embs]
            max_score = max(scores) if scores else 0

            thr = thresholds.get("USE", 0.65)
            pred = max_score > thr

            if pred and true_label:
                metrics["USE"]["TP"] += 1
            elif pred and not true_label:
                metrics["USE"]["FP"] += 1
            elif not pred and not true_label:
                metrics["USE"]["TN"] += 1
            else:
                metrics["USE"]["FN"] += 1

        except Exception as e:
            print("USE error:", e)

        # -------------------------
        # SBERT models
        # -------------------------
        for mid, model in sbert_models.items():
            try:
                resp_emb = model.encode(text)
                ref_embs = model.encode(answer_texts)

                scores = [cosine_similarity(resp_emb, ref) for ref in ref_embs]
                max_score = max(scores) if scores else 0

                thr = thresholds.get(mid, 0.65)
                pred = max_score > thr

                if pred and true_label:
                    metrics[mid]["TP"] += 1
                elif pred and not true_label:
                    metrics[mid]["FP"] += 1
                elif not pred and not true_label:
                    metrics[mid]["TN"] += 1
                else:
                    metrics[mid]["FN"] += 1

            except Exception as e:
                print(f"Error with model {mid}: {e}")

    # -------------------------
    # Final metrics
    # -------------------------
    for model_name, m in metrics.items():
        TP, FP, TN, FN = m["TP"], m["FP"], m["TN"], m["FN"]
        total = TP + FP + TN + FN

        acc = (TP + TN) / total if total else 0
        prec = TP / (TP + FP) if (TP + FP) else 0
        rec = TP / (TP + FN) if (TP + FN) else 0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0

        print(f"\n=== {model_name} ===")
        print(f"Accuracy:  {acc:.3f}")
        print(f"Precision: {prec:.3f}")
        print(f"Recall:    {rec:.3f}")
        print(f"F1 Score:  {f1:.3f}")

def evaluate_all_models(thresholds=None, log_filename="evaluation_log.txt"):
    if thresholds is None:
        thresholds = {}

    # -------------------------
    # Logger setup
    # -------------------------
    log_file = open(log_filename, "w", encoding="utf-8")

    def log(msg):
        print(msg)  # still show in Colab
        log_file.write(msg + "\n")

    # -------------------------
    # Load data ONCE
    # -------------------------
    # responses = supabase.table("responses").select("*").execute().data
    responses = (
        supabase
        .table("responses")
        .select("*")
        .range(1000, 2000)   # safely covers the remaining 450
        .execute()
        .data
    )
    answers_all = supabase.table("answers").select("*").execute().data

    # Group answers by question_id
    answers_by_qid = {}
    for a in answers_all:
        answers_by_qid.setdefault(a["question_id"], []).append(a["text"])

    # -------------------------
    # Metrics storage
    # -------------------------
    metrics = {}

    model_names = ["USE"] + list(sbert_models.keys())
    for m in model_names:
        metrics[m] = {"TP": 0, "FP": 0, "TN": 0, "FN": 0}

    # -------------------------
    # Evaluation loop
    # -------------------------
    for i, row in enumerate(responses):

        if i % 50 == 0:
            log(f"\nProcessed {i}/{len(responses)}")

        qid = row["question_id"]
        text = row["response_text"]
        true_label = row["meaning_label"]

        answer_texts = answers_by_qid.get(qid, [])
        if not answer_texts:
            continue

        # =========================
        # PRINT BASE INFO
        # =========================
        log("\n" + "="*100)
        log(f"Question ID: {qid}")
        log(f"Response: {text}")
        log("\nReference Answers:")
        for idx, ans in enumerate(answer_texts):
            log(f"  [{idx}] {ans}")

        # -------------------------
        # USE model
        # -------------------------
        try:
            resp_emb = use_embed([text])[0]
            ref_embs = use_embed(answer_texts)

            scores = [cosine_similarity(resp_emb, ref) for ref in ref_embs]

            log("\n--- USE Similarities ---")
            for idx, score in enumerate(scores):
                log(f"Ref[{idx}] similarity: {score:.6f}")

            max_score = max(scores) if scores else 0
            thr = thresholds.get("USE", 0.65)
            pred = max_score > thr

            if pred and true_label:
                metrics["USE"]["TP"] += 1
            elif pred and not true_label:
                metrics["USE"]["FP"] += 1
            elif not pred and not true_label:
                metrics["USE"]["TN"] += 1
            else:
                metrics["USE"]["FN"] += 1

        except Exception as e:
            log(f"USE error: {e}")

        # -------------------------
        # SBERT models
        # -------------------------
        for mid, model in sbert_models.items():
            try:
                resp_emb = model.encode(text)
                ref_embs = model.encode(answer_texts)

                scores = [cosine_similarity(resp_emb, ref) for ref in ref_embs]

                log(f"\n--- {mid} Similarities ---")
                for idx, score in enumerate(scores):
                    log(f"Ref[{idx}] similarity: {score:.6f}")

                max_score = max(scores) if scores else 0
                thr = thresholds.get(mid, 0.65)
                pred = max_score > thr

                if pred and true_label:
                    metrics[mid]["TP"] += 1
                elif pred and not true_label:
                    metrics[mid]["FP"] += 1
                elif not pred and not true_label:
                    metrics[mid]["TN"] += 1
                else:
                    metrics[mid]["FN"] += 1

            except Exception as e:
                log(f"Error with model {mid}: {e}")

    # -------------------------
    # Final metrics
    # -------------------------
    log("\n" + "="*100)
    log("FINAL METRICS")

    for model_name, m in metrics.items():
        TP, FP, TN, FN = m["TP"], m["FP"], m["TN"], m["FN"]
        total = TP + FP + TN + FN

        acc = (TP + TN) / total if total else 0
        prec = TP / (TP + FP) if (TP + FP) else 0
        rec = TP / (TP + FN) if (TP + FN) else 0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0

        log(f"\n=== {model_name} ===")
        log(f"Accuracy:  {acc:.3f}")
        log(f"Precision: {prec:.3f}")
        log(f"Recall:    {rec:.3f}")
        log(f"F1 Score:  {f1:.3f}")

    log_file.close()
"""

In [ ]:

evaluate_all_models()

from google.colab import files
files.download("evaluation_results.csv")

Loaded 1000 responses...
Loaded 1450 responses...
Loaded 155 answers...
Processed 0/1450
Processed 50/1450
Processed 100/1450
Processed 150/1450
Processed 200/1450
Processed 250/1450
Processed 300/1450
Processed 350/1450
Processed 400/1450
Processed 450/1450
Processed 500/1450
Processed 550/1450
Processed 600/1450
Processed 650/1450
Processed 700/1450
Processed 750/1450
Processed 800/1450
Processed 850/1450
Processed 900/1450
Processed 950/1450
Processed 1000/1450
Processed 1050/1450
Processed 1100/1450
Processed 1150/1450
Processed 1200/1450
Processed 1250/1450
Processed 1300/1450
Processed 1350/1450
Processed 1400/1450

Saved similarity results to: evaluation_results.csv

FINAL METRICS

=== USE ===
Accuracy : 0.6441
Precision: 0.8498
Recall   : 0.3454
F1 Score : 0.4911

=== sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ===
Accuracy : 0.6586
Precision: 0.7032
Recall   : 0.5423
F1 Score : 0.6124

=== sentence-transformers/LaBSE ===
Accuracy : 0.6641
Precision: 0.8162
Reca

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
evaluate_all_models(log_filename="evaluation_log_part2.txt")
from google.colab import files
files.download("evaluation_log_part2.txt")

Streaming output truncated to the last 5000 lines.
Ref[2] similarity: 0.477474

--- sentence-transformers/LaBSE Similarities ---
Ref[0] similarity: 0.483637
Ref[1] similarity: 0.455325
Ref[2] similarity: 0.478099

--- sentence-transformers/distiluse-base-multilingual-cased-v1 Similarities ---
Ref[0] similarity: 0.223091
Ref[1] similarity: 0.283315
Ref[2] similarity: 0.241934

--- akhooli/Arabic-SBERT-100K Similarities ---
Ref[0] similarity: 0.497612
Ref[1] similarity: 0.348715
Ref[2] similarity: 0.642912

--- Omartificial-Intelligence-Space/mmbert-base-arabic-nli Similarities ---
Ref[0] similarity: 0.718240
Ref[1] similarity: 0.568094
Ref[2] similarity: 0.673970

--- NAMAA-Space/AraModernBert-Base-STS Similarities ---
Ref[0] similarity: 0.515358
Ref[1] similarity: 0.377373
Ref[2] similarity: 0.631012

--- Omartificial-Intelligence-Space/Arabic-MiniLM-L12-v2-all-nli-triplet Similarities ---
Ref[0] similarity: 0.854604
Ref[1] similarity: 0.845918
Ref[2] similarity: 0.853822

--- Omartifi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 📊 Model Evaluation Summary (Arabic Semantic Similarity)

## 🎯 Objective

The goal of this experiment is to evaluate multiple sentence embedding models (including **USE** and **Sentence-BERT models**) on their ability to determine whether a student’s response matches the meaning of a reference answer.

Each response in the dataset has a ground-truth label:

* **True** → correct meaning
* **False** → incorrect meaning

---

## ⚙️ Methodology

For each response:

1. **Retrieve reference answers** corresponding to the same `question_id`.
2. **Generate embeddings**:

   * Using Universal Sentence Encoder Multilingual (USE)
   * Using multiple Sentence-BERT models
3. **Compute similarity**:

   * Cosine similarity between the response and each reference answer
4. **Select best match**:

   * Take the **maximum similarity score**
5. **Make prediction**:

   * If similarity > threshold (default = 0.65) → predicted **True (correct)**
   * Otherwise → predicted **False (incorrect)**
6. **Compare with ground truth** (`meaning_label`)

---

## 🧮 Evaluation Metrics

We evaluate each model using a **confusion matrix**:

* **TP (True Positive)**: predicted correct AND actually correct
* **TN (True Negative)**: predicted incorrect AND actually incorrect
* **FP (False Positive)**: predicted correct but actually incorrect
* **FN (False Negative)**: predicted incorrect but actually correct

---

## 📊 Metrics Explained

### 1. Accuracy

```text
Accuracy = (TP + TN) / Total
```

✔ Measures overall correctness
⚠️ Can be misleading if classes are imbalanced

---

### 2. Precision

```text
Precision = TP / (TP + FP)
```

✔ Measures how reliable “correct” predictions are
👉 High precision = few false positives

---

### 3. Recall

```text
Recall = TP / (TP + FN)
```

✔ Measures how many correct answers the model successfully detects
👉 High recall = fewer missed correct answers

---

### 4. F1 Score

```text
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```

✔ Balanced measure between precision and recall
👉 Best metric for overall model quality

---

## 🧠 Interpretation of Results

Different models behave differently:

* Some models (like USE) have **high precision but low recall**
  → Very strict: correct when confident, but misses many valid answers

* Some models have **high recall but low precision**
  → Very permissive: catches most correct answers but makes more mistakes

* The best models achieve a **balance (high F1 score)**

---

## 🏆 Key Takeaways

* Sentence embedding models can effectively measure **semantic similarity in Arabic**
* Model performance varies depending on:

  * strictness vs flexibility
  * multilingual vs Arabic-specific training
* **F1 score is the most reliable metric** for comparing models
* A combination of models (ensemble) could further improve performance

---

## 🚀 Conclusion

This evaluation framework allows us to:

* Benchmark multiple NLP models on Arabic semantic understanding
* Quantify their performance using standard metrics
* Identify the most suitable model for real-world deployment

---


In [ ]:
result = check_response_all_models(1, "لقد ظن أن الأطفال سيسخرون منه")
result = check_response_all_models(1, "لأن أدم طالب جديد وضعيف")


=== Model: USE (universal-sentence-encoder/3) ===
Comparisons for response: لقد ظن أن الأطفال سيسخرون منه
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  USE similarity:   0.795
Reference answer 2: لقد توقع أن يسخر منه الأطفال
  USE similarity:   0.787
Reference answer 3: لأن سالم طالب جديد وضعيف
  USE similarity:   0.158
Final decision (USE): Correct (score=0.795)

=== Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ===
Comparisons for response: لقد ظن أن الأطفال سيسخرون منه
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  SBERT similarity: 0.709
Reference answer 2: لقد توقع أن يسخر منه الأطفال
  SBERT similarity: 0.947
Reference answer 3: لأن سالم طالب جديد وضعيف
  SBERT similarity: 0.331
Final decision (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): Correct (score=0.947)

=== Model: sentence-transformers/LaBSE ===
Comparisons for response: لقد ظن أن الأطفال سيسخرون منه
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  SBERT similarity: 0.771
Ref

In [ ]:
result = check_response_all_models(2, "لأنهما يحبان كرة القدم ويستمتعان في اللعب معا")
result = check_response_all_models(2, "لأنهما يريدان معرفة من الاكثر اتقانا للعب")


=== Model: USE (universal-sentence-encoder/3) ===
Comparisons for response: لأنهما يحبان كرة القدم ويستمتعان في اللعب معا
Reference answer 1: لكي يستمتعوا معاً
  USE similarity:   0.328
Reference answer 2: لأن آدم وسالم يحبون لعب كرة القدم
  USE similarity:   0.589
Reference answer 3: لكي يتعرفوا على بعض
  USE similarity:   0.145
Reference answer 4: ليرى كل منهما من يتقنها أكثر
  USE similarity:   0.156
Final decision (USE): Incorrect (score=0.589)

=== Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ===
Comparisons for response: لأنهما يحبان كرة القدم ويستمتعان في اللعب معا
Reference answer 1: لكي يستمتعوا معاً
  SBERT similarity: 0.616
Reference answer 2: لأن آدم وسالم يحبون لعب كرة القدم
  SBERT similarity: 0.709
Reference answer 3: لكي يتعرفوا على بعض
  SBERT similarity: 0.334
Reference answer 4: ليرى كل منهما من يتقنها أكثر
  SBERT similarity: 0.308
Final decision (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): Correct (score=0.709)

=== Model: s

In [ ]:
result = check_response_all_models(1, "ليش يعني سخروا منه الاطفال") # False, (from responses table)
result = check_response_all_models(1, "خوفا من يسخر منه الاطفال") # True, (from responses table)


=== Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ===
Comparisons for response: ليش يعني سخروا منه الاطفال
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  SBERT similarity: 0.198
Reference answer 2: لقد توقع أن يسخر منه الأطفال
  SBERT similarity: 0.270
Reference answer 3: لأن سالم طالب جديد وضعيف
  SBERT similarity: 0.397
Final decision (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): Incorrect (score=0.397)

=== Model: sentence-transformers/LaBSE ===
Comparisons for response: ليش يعني سخروا منه الاطفال
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  SBERT similarity: 0.583
Reference answer 2: لقد توقع أن يسخر منه الأطفال
  SBERT similarity: 0.610
Reference answer 3: لأن سالم طالب جديد وضعيف
  SBERT similarity: 0.182
Final decision (sentence-transformers/LaBSE): Incorrect (score=0.610)

=== Model: sentence-transformers/distiluse-base-multilingual-cased-v1 ===
Comparisons for response: ليش يعني سخروا منه الاطفال
Reference answer 1: لقد خشي أن يسخ

In [ ]:
result = check_response_all_models(2, "الجايه لعبوا معي ") # False based on responses table
result = check_response_all_models(2, "حتى يلعب معه") # True based on responses table


=== Model: USE (universal-sentence-encoder/3) ===
Comparisons for response: الجايه لعبوا معي 
Reference answer 1: لكي يستمتعوا معاً
  USE similarity:   0.335
Reference answer 2: لأن آدم وسالم يحبون لعب كرة القدم
  USE similarity:   0.226
Reference answer 3: لكي يتعرفوا على بعض
  USE similarity:   0.233
Reference answer 4: ليرى كل منهما من يتقنها أكثر
  USE similarity:   0.121
Final decision (USE): Incorrect (score=0.335)

=== Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ===
Comparisons for response: الجايه لعبوا معي 
Reference answer 1: لكي يستمتعوا معاً
  SBERT similarity: 0.566
Reference answer 2: لأن آدم وسالم يحبون لعب كرة القدم
  SBERT similarity: 0.349
Reference answer 3: لكي يتعرفوا على بعض
  SBERT similarity: 0.451
Reference answer 4: ليرى كل منهما من يتقنها أكثر
  SBERT similarity: 0.377
Final decision (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): Incorrect (score=0.566)

=== Model: sentence-transformers/LaBSE ===
Comparisons for respons

In [ ]:
result = check_response_all_models(1, "لأن سالم طالب ضعيف وجديد")
result = check_response_all_models(2, "حتى يلعب معه")


=== Model: USE (universal-sentence-encoder/3) ===
Comparisons for response: لأن سالم طالب ضعيف وجديد
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  USE similarity:   0.210
Reference answer 2: لقد توقع أن يسخر منه الأطفال
  USE similarity:   0.084
Reference answer 3: لأن سالم طالب جديد وضعيف
  USE similarity:   0.849
Final decision (USE): Correct (score=0.849)

=== Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ===
Comparisons for response: لأن سالم طالب ضعيف وجديد
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  SBERT similarity: 0.233
Reference answer 2: لقد توقع أن يسخر منه الأطفال
  SBERT similarity: 0.298
Reference answer 3: لأن سالم طالب جديد وضعيف
  SBERT similarity: 0.825
Final decision (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): Correct (score=0.825)

=== Model: sentence-transformers/LaBSE ===
Comparisons for response: لأن سالم طالب ضعيف وجديد
Reference answer 1: لقد خشي أن يسخر منه الأطفال
  SBERT similarity: 0.177
Reference answer 2

## Sentence-BERT is a modification of BERT where the model is fine-tuned using a Siamese/triplet structure to directly produce sentence-level embeddings optimized for similarity tasks, rather than relying on token-level outputs and pooling strategies like averaging.

**1. sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2**

- Type: Multilingual SBERT
- Architecture: MiniLM
- Languages: 50+ languages (including Arabic)
- Arabic support:
    - ✔ Good
    - ✔ Trained on multilingual paraphrases
    - ❌ Not Arabic-specific fine-tuning
- Strength:
    - Fast, decent general semantic similarity
- Weakness:
    - Misses fine Arabic meaning differences (names, logic)
_______________________________________________________________
**2. sentence-transformers/LaBSE**
- Google model: Language-Agnostic BERT Sentence Embedding
- Arabic support:
    - ✔ Strong Arabic coverage
    - ✔ Trained on translation pairs across 100+ languages
- Strength:
    - Very strong multilingual alignment
- Weakness:
    - Still semantic, not “correctness-aware”
________________________________________________________________
**3. sentence-transformers/distiluse-base-multilingual-cased-v1**
- Based on USE + distillation
- Arabic support:
    - ✔ Yes (multilingual USE variant)
- Strength:
    - Stable embeddings
- Weakness:
    - Same USE issue: overly high similarity bias
_________________________________________________________________
**4. akhooli/Arabic-SBERT-100K**
- Arabic-focused SBERT
- Trained on Arabic sentence pairs (100K dataset)
- Arabic support:
    - ✔ Strong Arabic specialization
- Strength:
    - Better Arabic semantic discrimination than generic models
- Weakness:
    - Still no factual reasoning
________________________________________________________________
**5. Omartificial-Intelligence-Space/mmbert-base-arabic-nli**
- Arabic NLI model (Natural Language Inference)
- Arabic support:
    - ✔ Strong Arabic NLI training
- Strength:
    - Better for:
         - entailment
         - contradiction detection
    - Best use:
         - ✔ grading correctness (better than SBERT)
_________________________________________________________________
**6. NAMAA-Space/AraModernBert-Base-STS**
- Arabic STS (Semantic Textual Similarity)
- Arabic support:
    - ✔ Arabic-first model
- Strength:
    - Best for Arabic similarity scoring
- Weakness:
    - Still similarity-only, not reasoning
_________________________________________________________________
**7. Omartificial-Intelligence-Space/Arabic-MiniLM-L12-v2-all-nli-triplet**
- Arabic MiniLM trained with NLI triplet loss
- Arabic support:
    - ✔ Strong Arabic training
- Strength:
    - Good balance of:
         - similarity + NLI signals
________________________________________________________________
**8. Omartificial-Intelligence-Space/Arabic-mpnet-base-all-nli-triplet**
- MPNet architecture + Arabic NLI training
- Arabic support:
    - ✔ Strong
- Strength:
    - One of the best Arabic sentence embedding models
- Weakness:
    - Still not strict correctness checker
__________________________________________________________________
**9. medmediani/Arabic-KW-Mdel**
- Arabic keyword / semantic model (likely STS / retrieval-focused)
- Arabic support:
    - ✔ Arabic-specific
- Strength:
    - Good keyword-aware similarity
- Weakness:
    - Still embedding-based, not logical checker
---


| **Model** | **Arabic Support** | **Strength** | **Weakness** | **Best Use** |
| --- | --- | --- | --- | --- |
| **paraphrase-multilingual-MiniLM-L12-v2** | **50+ languages incl. Arabic** | **Fast; good general semantic similarity** | **Not Arabic‑fine tuned; misses subtle Arabic nuances** | **Cross‑lingual retrieval; fast embedding pipelines**. |
| **LaBSE** | **100+ languages; strong Arabic** | **Very strong multilingual alignment** | **Not a factual correctness checker** | **Cross‑lingual matching; translation‑pair alignment**. |
| **distiluse-base-multilingual-cased-v1** | **Multilingual USE variant incl. Arabic** | **Stable, consistent embeddings** | **Tends to overestimate similarity** | **Robust retrieval; legacy USE workflows**. |
| **akhooli/Arabic-SBERT-100K** | **Arabic‑specialized** | **Better Arabic semantic discrimination** | **No factual reasoning** | **Arabic STS and clustering** |
| **Omartificial-Intelligence-Space/mmbert-base-arabic-nli** | **Arabic NLI training** | **Good at entailment/contradiction** | **Not general embedding for search** | **Grading correctness; contradiction detection** |
| **NAMAA-Space/AraModernBert-Base-STS** | **Arabic‑first STS** | **Top Arabic similarity scoring** | **Similarity only, not reasoning** | **Fine‑grained Arabic STS** |
| **Omartificial-Intelligence-Space/Arabic-MiniLM-L12-v2-all-nli-triplet** | **Arabic + NLI triplet** | **Similarity + NLI signal balance** | **Still embedding‑based** | **Similarity with some correctness signal** |
| **Omartificial-Intelligence-Space/Arabic-mpnet-base-all-nli-triplet** | **Arabic + MPNet architecture** | **One of best Arabic sentence embeddings** | **Not a strict logical verifier** | **High‑quality Arabic embeddings for STS** |
| **medmediani/Arabic-KW-Mdel** | **Arabic keyword focus** | **Keyword‑aware similarity** | **Embedding limitations for logic** | **Keyword retrieval and intent matching** |

---

# 🧠 Universal Sentence Encoder Multilingual (USE M-3)

Universal Sentence Encoder Multilingual is a pre-trained neural network that converts **sentences (not words)** into fixed-size numerical vectors (embeddings), so you can compare meaning across languages like Arabic, English, French, etc.

---

# 🌍 What it does

It takes a sentence like:

* Arabic: **"أنا أحب البرمجة"**
* English: **"I love programming"**

And converts each one into a vector such as:

```text
[0.12, -0.44, 0.87, ...]   (512-dimensional vector)
```

Sentences with similar meaning → vectors close together.

---

# ⚙️ How it works (core idea)

## 1. Input sentence

You give it text (any supported language, including Arabic).

---

## 2. Tokenization + encoding

Internally, the model:

* splits text into subwords
* uses a neural encoder (Transformer or DAN depending on version)
* processes the full sentence with context

---

## 3. Sentence embedding creation

Instead of outputting word vectors, it produces:

> ✅ ONE single vector representing the entire sentence meaning

This vector is typically **512 dimensions**.

---

## 4. Training objective (important part)

It is trained using tasks like:

* translation pairs (Arabic ↔ English)
* paraphrase detection
* semantic similarity datasets
* question-answer matching

So it learns:

> sentences with same meaning → closer vectors
> different meaning → farther vectors

---

# 🌐 Why it supports Arabic

The model is **multilingual**, trained on 16 languages including Arabic.

That means it learns a shared “semantic space” where:

* Arabic sentences
* English sentences
* French sentences

can all be compared directly.

---

# 📊 How similarity works

Once you have embeddings:

```python
cosine_similarity(A, B)
```

* High score → same meaning
* Low score → different meaning

Example:

| Sentence                              | Similarity |
| ------------------------------------- | ---------- |
| "أنا أحب القطط" vs "أحب القطط كثيراً" | 🔼 high    |
| "أنا أحب القطط" vs "الطقس جميل اليوم" | 🔽 low     |

---

# 🧠 Key design idea

USE is different from BERT/SBERT:

### Unlike BERT:

* ❌ not token-focused
* ❌ not meant for word embeddings

### Unlike SBERT:

* ✔ trained directly as a sentence encoder
* ✔ no need for extra fine-tuning for basic similarity tasks

---

# 🏗️ Architecture (simplified)

It uses one of two encoders depending on variant:

### 1. Transformer-based USE (modern versions)

* attention mechanism
* context-aware encoding
* better accuracy

### 2. Deep Averaging Network (older versions)

* average word embeddings
* feed-forward network
* faster but weaker

---

# ⚡ Why people use it

* very easy to use (`model(sentences)`)
* strong baseline for semantic similarity
* works cross-language (Arabic included)
* good for clustering, search, classification

---

# 🚨 Limitations (important for your use case)

* weaker than modern SBERT models for Arabic nuance
* older architecture compared to transformer-only embeddings
* some versions (like Kaggle “large”) break in modern Python setups

---

# 🧭 Simple mental model

Think of USE as:

> “A universal meaning fingerprint generator for sentences”

Instead of analyzing grammar or words, it directly maps meaning → vector space.

---

# 👍 Bottom line

* ✔ Supports Arabic
* ✔ Produces sentence embeddings directly
* ✔ Trained for cross-language meaning alignment
* ✔ Easy similarity comparison with cosine similarity

But:

> ⚠️ Not as strong as modern Sentence-BERT models for Arabic nuance


USE variants (like `distiluse‑base‑multilingual‑cased‑v1`) produce stable, “legacy” sentence vectors that often **over‑estimate** similarity, while Sentence‑BERT models produce more discriminative embeddings and are easier to fine‑tune for Arabic nuance; for strict Arabic STS or grading, Arabic‑fine‑tuned SBERT/NLI models usually give better real‑world accuracy.**   [amitness.com](https://amitness.com/posts/universal-sentence-encoder)  [Sentence-Transformers](https://www.sbert.net/)

### How they work — plain examples
**Universal Sentence Encoder style (USE)**  
- **Mechanics**: USE learns fixed‑length vectors from a mix of unsupervised and supervised objectives so that semantically similar sentences map close together.   [amitness.com](https://amitness.com/posts/universal-sentence-encoder)  
- **Arabic behavior example**:  
  - Input A: **"ذهبت إلى المدرسة"**  
  - Input B: **"ذهبت للمدرسة اليوم"**  
  - **Expected**: high similarity (both mean going to school). USE will map them very close; it may also map **"ذهبت إلى السوق"** fairly close because of general travel/visit semantics — **over‑similarity** risk.   [amitness.com](https://amitness.com/posts/universal-sentence-encoder)

**Sentence‑BERT family (SBERT, LaBSE, MPNet variants)**  
- **Mechanics**: SBERT fine‑tunes transformer encoders with siamese/triplet or NLI losses so cosine distance better reflects sentence‑level semantics and paraphrase relations. LaBSE is a Google BERT variant trained on translation pairs for language‑agnostic alignment.   [Sentence-Transformers](https://www.sbert.net/)  [arXiv.org](https://arxiv.org/abs/2309.03747)  
- **Arabic behavior example**:  
  - Input A: **"الرئيس وقع القانون"**  
  - Input B1: **"الرئيس لم يوقع القانون"** (negation)  
  - Input B2: **"الرئيس وقع الاتفاق"** (different object)  
  - **Expected**: SBERT/NLI‑trained models separate B1 (contradiction) and B2 (different but related). SBERT variants tuned on Arabic triplets or NLI do this better than USE.   [Hugging Face](https://huggingface.co/akhooli/Arabic-SBERT-100K)  [Github](https://github.com/m-elbeltagi/Comparing_Arabic_Sentence_Transformers)

### Evidence on Arabic accuracy
- **Empirical trend**: Arabic‑specialized SBERT models (trained on Arabic triplets or NLI) outperform generic multilingual encoders on Arabic STS and classification tasks; community benchmarks and model cards show improved discrimination and lower false positives.   [Hugging Face](https://huggingface.co/akhooli/Arabic-SBERT-100K)  [Github](https://github.com/m-elbeltagi/Comparing_Arabic_Sentence_Transformers)  
- **Caveat**: recent analyses show all sentence encoders can fail basic semantic properties despite good benchmark scores — so test on your Arabic validation set.   [arXiv.org](https://arxiv.org/abs/2309.03747)

### Compact comparison table
| **Model Family** | **Core Architecture** | **Arabic Strength** | **Best For** | **Key Note** |
|---|---:|---:|---|---|
| **USE variant** | Distilled Transformer | **Good but over‑similar** | Stable retrieval; legacy pipelines | Tends to map related but distinct sentences too close. |
| **SBERT (MiniLM/MPNet)** | Siamese fine‑tuned BERT | **Better Arabic nuance** | STS, paraphrase, clustering | Easier to fine‑tune on Arabic triplets. |
| **LaBSE** | Language‑agnostic BERT | **Strong cross‑lingual Arabic** | Arabic↔English matching | Excellent translation alignment. |
| **Arabic‑NLI models** | BERT/MPNet + NLI loss | **Best for correctness** | Entailment, grading, contradiction | Use for grading correctness rather than pure similarity. |


For **Comprehension Assessment (Arabic children answers)** task, we are essentially doing:

> sentence → semantic embedding → similarity scoring → correctness decision

So we need models that are:

* strong in **Arabic**
* stable for **short sentence meaning**
* not overly “overconfident” like USE
* reasonably robust for grading-like behavior

---

# 🥇 BEST CHOICE (from previous 9 models)

## ✅ 1. LaBSE — **sentence-transformers/LaBSE**

### Why:

* Trained on **100+ languages including Arabic**
* Designed to align translations → very strong semantic alignment
* Very stable for sentence-level similarity
* Less biased than USE (more conservative scores)

### For the task:
- ✔ Good at “meaning equivalence”
- ✔ Works well with short Arabic answers
- ✔ Better separation than USE in many cases

---

# 🥈 2. Best Arabic-specialized model

## ✅ NAMAA-Space/AraModernBert-Base-STS

### Why:

* Specifically trained for **Arabic STS (Semantic Textual Similarity)**
* Much more sensitive to Arabic sentence meaning than multilingual models
* Better at subtle differences in Arabic phrasing

### For the task:

- ✔ Better Arabic nuance
- ✔ Better than generic SBERT for Arabic grading
- ✔ Good second model for comparison

---

# 🥉 3. Best “Arabic SBERT-style” model

## ✅ Omartificial-Intelligence-Space/Arabic-mpnet-base-all-nli-triplet

### Why:

* MPNet backbone (stronger than MiniLM)
* Fine-tuned on **NLI + triplet loss**
* Better semantic structure understanding than MiniLM

### For the task:

- ✔ Good balance of similarity + meaning
- ✔ Better ranking consistency
- ✔ Strong second/third opinion model

---

# 👍 4. Good lightweight Arabic model

## Omartificial-Intelligence-Space/Arabic-MiniLM-L12-v2-all-nli-triplet

### Why:

* Fast and efficient
* Arabic-specific training
* NLI-aware (better than pure similarity models)

### Limitation:

* Less accurate than mpnet / LaBSE

---

# 👍 5. akhooli/Arabic-SBERT-100K

### Why:

* Pure Arabic SBERT model
* Trained on Arabic sentence pairs

### Strength:

- ✔ Simple and stable
- ✔ Good baseline Arabic similarity

### Weakness:

* Older / smaller training quality compared to newer NLI models

---

# ⚠️ 6. mmbert-base-arabic-nli

### Why:

* NLI-based Arabic model (important for correctness tasks)

### Strength:

- ✔ Best model in your list for “correct vs incorrect reasoning”
- ✔ Can help detect contradictions better than SBERT

### BUT:

* Not purely optimized for similarity scoring stability

👉 Use it as a **decision layer**, not just similarity

---

# ⚠️ 7. distiluse-base-multilingual-cased-v1

### Why:

* USE-based distillation model

### Problem:

* Has same issue as USE:
  👉 very high similarity even for wrong answers

- ✔ OK baseline
- ❌ Not good for grading

---

# ⚠️ 8. MiniLM multilingual

### Why:

* Fast, widely used

### Problem:

* Weak Arabic nuance compared to LaBSE / Arabic models

---

# ⚠️ 9. Arabic-KW-Mdel

### Why:

* Arabic semantic/keyword-oriented model

### Problem:

* Less consistent for sentence-level grading

---

# 🧠 FINAL RECOMMENDATION (VERY IMPORTANT)

For your research system, do NOT use only one model.

## 🏆 Best architecture for your paper:

### ✔ Primary model (semantic similarity)

* **LaBSE**

### ✔ Arabic specialist model

* **NAMAA-AraModernBERT-STS**

### ✔ Arabic NLI / reasoning model

* **mmbert-base-arabic-nli**

### ✔ Backup / lightweight comparison

* **Arabic mpnet triplet model**

---

# 🔥 WHY THIS COMBINATION WORKS

Because your task is NOT just similarity:

You need:

| Ability                     | Model type    |
| --------------------------- | ------------- |
| Meaning similarity          | LaBSE         |
| Arabic nuance               | AraModernBERT |
| correctness / contradiction | NLI model     |
| robustness                  | mpnet / SBERT |

---

# ⚠️ IMPORTANT INSIGHT (based on your earlier bug)

Your failure case:

> “أدم” vs “سالم”

This is NOT a similarity problem.

It is:

> entity mismatch / factual inconsistency

👉 So even the best SBERT models will fail unless you add:

* rule-based entity checking OR
* NLI model layer

🎯 What the LLM needs
To generate good feedback, send:

✅ Required inputs


Question


User response


Reference answer(s)


Similarity score


Prediction (correct / incorrect)



✨ Example prompt
You are an assistant helping evaluate student answers.

Question:
{question}

Reference Answer:
{best_reference_answer}

Student Answer:
{user_response}

Similarity Score: {score}
Prediction: {correct / incorrect}

Task:
- Explain whether the student's answer is correct or not
- If incorrect, explain what is missing or wrong
- If partially correct, explain what is good and what needs improvement
- Keep feedback short and clear